## Tensor Flow



#### First we using a  tf 1.0

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()



Instructions for updating:
non-resource variables are not supported in the long term


### Linear Regression

In [6]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

m , n = housing.data.shape

housing_data_plus_bias = np.c_[np.ones((m, 1)), housing.data]
scaled_housing_data_plus_bias = np.c_[np.ones((m, 1)), (housing.data - housing.data.mean(axis=0)) / housing.data.std(axis=0)]

X = tf.constant(housing_data_plus_bias, dtype=tf.float32, name="X")
y = tf.constant(housing.target.reshape(-1,1), dtype=tf.float32, name="y")

XT = tf.transpose(X)

theta = tf.matmul(tf.matmul(tf.linalg.inv(tf.matmul(XT, X)), XT), y)
with tf.Session() as sess:
    theta_value = theta.eval()


### Implementing Gradient Descent


In [7]:
n_epochs = 1000
learning_rate = 0.01


x = tf.constant(scaled_housing_data_plus_bias, dtype=tf.float32, name="x" )
y = tf.constant(housing.target.reshape(-1,1), dtype=tf.float32, name="y" )  


theta = tf.Variable(tf.random_uniform([n+1,1], -1.0, 1.0), name="theta" )

y_pred = tf.matmul(x, theta, name="predictions" )       
error = y_pred - y

mse = tf.reduce_mean(tf.square(error), name="mse" ) 
gradients   = 2/m * tf.matmul(tf.transpose(x), error)
training_op = tf.assign(theta, theta - learning_rate * gradients)   
init = tf.global_variables_initializer()    



with tf.Session() as sess :

    sess.run(init) 
    for epoch in range(n_epochs):
        if epoch % 100 == 0:
            print("Epoch", epoch, "MSE =", mse.eval())
        sess.run(training_op)

    best_theta = theta.eval()    



Epoch 0 MSE = 9.37464
Epoch 100 MSE = 0.6871553
Epoch 200 MSE = 0.53714335
Epoch 300 MSE = 0.53215927
Epoch 400 MSE = 0.5305278
Epoch 500 MSE = 0.52930707
Epoch 600 MSE = 0.52834386
Epoch 700 MSE = 0.5275781
Epoch 800 MSE = 0.5269666
Epoch 900 MSE = 0.526476


### Auto diff

In [8]:
# Tf automaticcally compute the gradients
gradients = tf.gradients(mse, [theta])[0] 
training_op = tf.assign(theta, theta - learning_rate * gradients)   


### Optimizer


In [9]:
#Uisng an optimizer
optimizer = tf.train.GradientDescentOptimizer(learning_rate=learning_rate)
training_op = optimizer.minimize(mse)

optimizer2  = tf.train.MomentumOptimizer(learning_rate=learning_rate, momentum=0.9)

### Feeding Data to the trsining algorithm


In [10]:
A  = tf.placeholder(tf.float32, shape=(None, 3), name="A" )
B =  A + 5

with tf.Session() as sess:

    B_val_1 = B.eval(feed_dict={A: [[1, 2, 3]]})
    B_val_2 = B.eval(feed_dict={A: [[4, 5, 6], [7, 8, 9]]}) 


    print(B_val_1)
    print(B_val_2)

[[6. 7. 8.]]
[[ 9. 10. 11.]
 [12. 13. 14.]]


In [11]:
# Implement Mini batch gradient descent

X = tf.placeholder(tf.float32, shape=(None, n+1), name="X" )
y = tf.placeholder(tf.float32, shape=(None, 1), name="y")

batch_size = 100
n_batches = int(np.ceil(m / batch_size))


def fetch_batch(epoch, batch_index, batch_size): 
    np.random.seed(epoch * n_batches + batch_index) 
    indices = np.random.randint(m, size=batch_size) 
    X_batch = scaled_housing_data_plus_bias[indices]
    y_batch = housing.target.reshape(-1, 1)[indices]
    return X_batch, y_batch
with tf.Session() as sess:
    sess.run(init) 
    for epoch in range(n_epochs):
        for batch_index in range(n_batches):
            X_batch, y_batch = fetch_batch(epoch, batch_index, batch_size)
            sess.run(training_op, feed_dict={X: X_batch, y: y_batch})
        best_theta = theta.eval()    

In [22]:
# Graph-safe mini-batch training
import tensorflow.compat.v1 as tf


X_data = np.random.rand(1000,10).astype(np.float32)
y_data = (np.sum(X_data, axis=1, keepdims=True) > 5).astype(np.float32)

batch_size = 100
num_samples = X_data.shape[0]
n_batches = int(np.ceil(num_samples / batch_size))
n_epochs = 10


def fetch_batch(epoch, batch_index, batch_size):
    np.random.seed(epoch * n_batches + batch_index)
    indices = np.random.randint(num_samples, size=batch_size)
    X_batch = X_data[indices]
    y_batch = y_data[indices]
    return X_batch, y_batch


tf.reset_default_graph()
X = tf.placeholder(tf.float32, shape=(None, X_data.shape[1]), name="X")
y = tf.placeholder(tf.float32, shape=(None, 1), name="y")

W = tf.Variable(tf.random_normal([X_data.shape[1], 1]), name="W")
b = tf.Variable(tf.zeros([1]), name="b")

logits = tf.matmul(X, W) + b
loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits))
training_op = tf.train.AdamOptimizer(learning_rate=0.01).minimize(loss)
init = tf.global_variables_initializer()


with tf.Session() as sess:
    sess.run(init)
    for epoch in range(n_epochs):
        for batch_index in range(n_batches):
            X_batch, y_batch = fetch_batch(epoch, batch_index, batch_size)
            sess.run(training_op, feed_dict={X: X_batch, y: y_batch})
        loss_value = sess.run(loss, feed_dict={X: X_batch, y: y_batch})
        print("Epoch:", epoch, "Loss:", loss_value)





Epoch: 0 Loss: 0.704544
Epoch: 1 Loss: 0.6472918
Epoch: 2 Loss: 0.75828904
Epoch: 3 Loss: 0.7843675
Epoch: 4 Loss: 0.71939796
Epoch: 5 Loss: 0.7053601
Epoch: 6 Loss: 0.6913
Epoch: 7 Loss: 0.67684853
Epoch: 8 Loss: 0.6802777
Epoch: 9 Loss: 0.66919607


### Saving and Restoring Models

In [ ]:
# Save model to disk
saver = tf.train.Saver()
save_path = saver.save(sess, "/tmp/my_model_final.ckpt")
print(f"Model saved to: {save_path}")

# Restore model from disk
tf.reset_default_graph()
X = tf.placeholder(tf.float32, shape=(None, X_data.shape[1]), name="X")
y = tf.placeholder(tf.float32, shape=(None, 1), name="y")

W = tf.Variable(tf.random_normal([X_data.shape[1], 1]), name="W")
b = tf.Variable(tf.zeros([1]), name="b")

logits = tf.matmul(X, W) + b
loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits))

saver = tf.train.Saver()

with tf.Session() as sess:
    saver.restore(sess, "/tmp/my_model_final.ckpt")
    print("Model restored successfully")
    
    # Make predictions with restored model
    predictions = sess.run(logits, feed_dict={X: X_data[:10]})
    print(f"Predictions: {predictions}")

### Visualizing the Graph and training curves using tensorboard

In [26]:
from datetime import datetime

# TensorBoard example: visualize graph + training curve (TensorFlow v1 style)

try:
    logdir
except NameError:
    now = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%S")
    root_logdir = "tf_logs"
    logdir = "{}/run-{}/".format(root_logdir, now)

tf.reset_default_graph()

X_tb = tf.placeholder(tf.float32, shape=(None, X_data.shape[1]), name="X_tb")
y_tb = tf.placeholder(tf.float32, shape=(None, 1), name="y_tb")

W_tb = tf.Variable(tf.random_normal([X_data.shape[1], 1]), name="W_tb")
b_tb = tf.Variable(tf.zeros([1]), name="b_tb")

logits_tb = tf.matmul(X_tb, W_tb) + b_tb
loss_tb = tf.reduce_mean(
    tf.nn.sigmoid_cross_entropy_with_logits(labels=y_tb, logits=logits_tb),
    name="loss_tb",
)

global_step_tb = tf.Variable(0, trainable=False, name="global_step_tb")
training_op_tb = tf.train.AdamOptimizer(learning_rate=0.01).minimize(
    loss_tb, global_step=global_step_tb
)

loss_summary_tb = tf.summary.scalar("batch_loss", loss_tb)
init_tb = tf.global_variables_initializer()

with tf.Session() as sess:
    sess.run(init_tb)
    file_writer = tf.summary.FileWriter(logdir, tf.get_default_graph())

    for epoch in range(n_epochs):
        for batch_index in range(n_batches):
            X_batch, y_batch = fetch_batch(epoch, batch_index, batch_size)
            _, summary_str, step_val = sess.run(
                [training_op_tb, loss_summary_tb, global_step_tb],
                feed_dict={X_tb: X_batch, y_tb: y_batch},
            )
            file_writer.add_summary(summary_str, step_val)

        epoch_loss = sess.run(loss_tb, feed_dict={X_tb: X_data, y_tb: y_data})
        print("Epoch:", epoch, "Loss:", epoch_loss)

    file_writer.flush()
    file_writer.close()

print("TensorBoard logdir:", logdir)
print("Run: tensorboard --logdir=tf_logs")

Epoch: 0 Loss: 0.7032378
Epoch: 1 Loss: 0.69580984
Epoch: 2 Loss: 0.68796253
Epoch: 3 Loss: 0.68002486
Epoch: 4 Loss: 0.67302835
Epoch: 5 Loss: 0.66538745
Epoch: 6 Loss: 0.657928
Epoch: 7 Loss: 0.65078884
Epoch: 8 Loss: 0.64417195
Epoch: 9 Loss: 0.6390858
TensorBoard logdir: tf_logs/run-2026-04-28T16-55-36/
Run: tensorboard --logdir=tf_logs


In [25]:
from datetime import datetime 
now = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%S")
root_logdir = "tf_logs"
logdir = "{}/run-{}/".format(root_logdir, now)


mse_summary = tf.summary.scalar('MSE', mse)
file_writer = tf.summary.FileWriter(logdir, tf.get_default_graph())

C:\Users\User\AppData\Local\Temp\ipykernel_21256\1710583910.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%S")


### Names Scopes

In [27]:
import numpy as np

# Example: using name scopes in TensorFlow v1 (compat.v1)

with tf.name_scope("Inputs"):
    x_ns = tf.placeholder(tf.float32, shape=(None, 3), name="x_ns")
    y_ns = tf.placeholder(tf.float32, shape=(None, 1), name="y_ns")

with tf.name_scope("Layer1"):
    W1 = tf.Variable(tf.random_normal([3, 4]), name="W1")
    b1 = tf.Variable(tf.zeros([4]), name="b1")
    z1 = tf.matmul(x_ns, W1) + b1
    a1 = tf.tanh(z1, name="a1")

with tf.name_scope("Layer2"):
    W2 = tf.Variable(tf.random_normal([4, 1]), name="W2")
    b2 = tf.Variable(tf.zeros([1]), name="b2")
    logits_ns = tf.matmul(a1, W2) + b2
    output = tf.sigmoid(logits_ns, name="output")

with tf.name_scope("Loss"):
    loss_ns = tf.reduce_mean(
        tf.nn.sigmoid_cross_entropy_with_logits(labels=y_ns, logits=logits_ns),
        name="loss"
    )

# initialize and run a single step to show op/tensor names
init_ns = tf.global_variables_initializer()
X_sample = np.random.rand(5, 3).astype(np.float32)
y_sample = (np.sum(X_sample, axis=1, keepdims=True) > 1.5).astype(np.float32)

with tf.Session() as sess2:
    sess2.run(init_ns)
    out_val, loss_val = sess2.run([output, loss_ns], feed_dict={x_ns: X_sample, y_ns: y_sample})

print("Tensor names:")
print(x_ns.name, W1.name, b1.name, a1.name, W2.name, output.name, loss_ns.name)
print("Output sample:\n", out_val)
print("Loss:", loss_val)

# add graph to existing FileWriter (if present) for TensorBoard visualization
try:
    file_writer.add_graph(tf.get_default_graph())
    file_writer.flush()
    print("Graph added to FileWriter at:", logdir)
except Exception:
    pass

Tensor names:
Inputs/x_ns:0 Layer1/W1:0 Layer1/b1:0 Layer1/a1:0 Layer2/W2:0 Layer2/output:0 Loss/loss:0
Output sample:
 [[0.514279  ]
 [0.31278205]
 [0.31865764]
 [0.2580399 ]
 [0.521385  ]]
Loss: 0.6859382
Graph added to FileWriter at: tf_logs/run-2026-04-28T16-55-36/


c:\Users\User\Music\deep_learning_projects\.venv\Lib\site-packages\tensorflow\python\summary\writer\writer.py:438: UserWarning: Attempting to use a closed FileWriter. The operation will be a noop unless the FileWriter is explicitly reopened.
  warnings.warn("Attempting to use a closed FileWriter. "
